# GLAD_eval_multi

MVTec-AD の複数カテゴリ（推奨: bottle, cable, capsule, pill, grid）を GLAD で評価するための notebook です。

## 使い方
1. ランタイムを **GPU** にする
2. Drive を mount する
3. `GLAD_CONFIG` の各カテゴリ設定を埋める
4. Drive 上の dataset / checkpoint / pretrained model を配置する
5. セルを上から順に実行する

## この notebook が保存するもの
- `comparison/GLAD/<category>/metrics.json`
- `comparison/GLAD/<category>/samples/`（可視化画像を保存する場合）
- `comparison/tables/glad_multi_metrics.csv`


## Stable Diffusion v1-4 について
- `git clone https://github.com/hyao1/GLAD.git` **だけでは** `stable-diffusion-v1-4` は取得されません。
- 初回は Hugging Face から `CompVis/stable-diffusion-v1-4` をダウンロードして、Drive の
  `industrial-anomaly-detection-app/pretrained/stable-diffusion-v1-4/` に保存してください。
- 下の「Stable Diffusion v1-4 の存在確認 / ダウンロード」セルで確認・取得できます。


In [1]:
# 0. GPU 確認
!nvidia-smi


Thu Mar 12 02:47:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Drive mount
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Stable Diffusion v1-4 の存在確認 / ダウンロード
GLAD の `git clone` では重みは入りません。
この notebook では、Drive 上の `industrial-anomaly-detection-app/pretrained/stable-diffusion-v1-4` を使います。
フォルダが無い場合は、次のセルで Hugging Face から保存してください。


In [3]:
# 1.5 Stable Diffusion v1-4 の保存先確認
import os

SD_MODEL_ID = "CompVis/stable-diffusion-v1-4"
SD_SAVE_DIR = "/content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/stable-diffusion-v1-4"

print("SD_SAVE_DIR:", SD_SAVE_DIR)
print("exists:", os.path.exists(SD_SAVE_DIR))
if os.path.exists(SD_SAVE_DIR):
    print("children:", os.listdir(SD_SAVE_DIR)[:20])
else:
    print("stable-diffusion-v1-4 がまだありません。次のセルでダウンロードできます。")


SD_SAVE_DIR: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/stable-diffusion-v1-4
exists: True
children: ['.cache', 'safety_checker', 'feature_extractor', 'scheduler', 'text_encoder', 'tokenizer', 'unet', 'vae', '.huggingface', 'model_index.json', '.gitattributes', 'README.md', 'v1-variants-scores.jpg']


In [4]:
# 1.6 Stable Diffusion v1-4 を Hugging Face から Drive に保存（必要なときだけ実行）
# 使い方:
# 1) Hugging Face で CompVis/stable-diffusion-v1-4 の利用規約に同意
# 2) このセルを実行して token を入力
# 3) 保存完了後、以降は再実行不要

RUN_SD_DOWNLOAD = True  # 必要なときだけ True にする

if RUN_SD_DOWNLOAD:
    !pip install -q huggingface_hub
    from huggingface_hub import login, snapshot_download
    os.makedirs(SD_SAVE_DIR, exist_ok=True)
    login()
    snapshot_download(
        repo_id=SD_MODEL_ID,
        local_dir=SD_SAVE_DIR,
        local_dir_use_symlinks=False,
    )
    print("saved to:", SD_SAVE_DIR)
else:
    print("skip download. 既に Drive にある場合はこのままで OK です。")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1194: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 33 files:   0%|          | 0/33 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

v1-variants-scores.jpg:   0%|          | 0.00/71.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

saved to: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/stable-diffusion-v1-4


In [5]:
# 2. 基本設定
import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/industrial-anomaly-detection-app"
DATA_ROOT = f"{BASE_DIR}/data/MVTec-AD"
PRETRAIN_ROOT = f"{BASE_DIR}/pretrained"
COMPARISON_ROOT = f"{BASE_DIR}/comparison"
GLAD_REPO_DIR = "/content/GLAD"

CATEGORIES = ["bottle", "cable", "capsule", "pill", "grid"]

# bottle は実測済みの値を入れてあります。
# 他カテゴリは checkpoint と推論パラメータを埋めてください。
GLAD_CONFIG = {
    "bottle": {
        "enabled": True,
        "dataset_path": f"{DATA_ROOT}/bottle",
        "pretrained_sd_path": f"{PRETRAIN_ROOT}/stable-diffusion-v1-4",
        "checkpoint_dir": "/content/GLAD/model/bottle_4000step_bs32_eps_anomaly2_0/checkpoint-2500",
        "checkpoint_step": 2500,
        "output_name": "bottle_4000step_bs32_eps_anomaly2_0",
        "denoise_step": 750,
        "input_threshold": 0.32,
        "v": 0,
        "min_step": 350,
        "resolution": 512,
        "test_batch_size": 2,
        "dino_resolution": 512,
        "inference_step": 25,
        "save_visuals": True,
    },
    "cable": {
        "enabled": True,
        "dataset_path": f"{DATA_ROOT}/cable",
        "pretrained_sd_path": f"{PRETRAIN_ROOT}/stable-diffusion-v1-4",
        "checkpoint_dir": "/content/GLAD/model/cable_4000step_bs32_eps_anomaly2_0/checkpoint-1500",
        "checkpoint_step": 1500,
        "output_name": "cable_4000step_bs32_eps_anomaly2_0",
        "denoise_step": 750,
        "input_threshold": 0.40,
        "v": 0,
        "min_step": 350,
        "resolution": 512,
        "test_batch_size": 2,
        "dino_resolution": 512,
        "inference_step": 25,
        "save_visuals": True,
    },
    "capsule": {
        "enabled": True,
        "dataset_path": f"{DATA_ROOT}/capsule",
        "pretrained_sd_path": f"{PRETRAIN_ROOT}/stable-diffusion-v1-4",
        "checkpoint_dir": "/content/GLAD/model/capsule_4000step_bs32_eps_anomaly2_0/checkpoint-2000",
        "checkpoint_step": 2000,
        "output_name": "capsule_4000step_bs32_eps_anomaly2_0",
        "denoise_step": 600,
        "input_threshold": 0.40,
        "v": 0,
        "min_step": 350,
        "resolution": 512,
        "test_batch_size": 2,
        "dino_resolution": 512,
        "inference_step": 25,
        "save_visuals": True,
    },
    "pill": {
        "enabled": True,
        "dataset_path": f"{DATA_ROOT}/pill",
        "pretrained_sd_path": f"{PRETRAIN_ROOT}/stable-diffusion-v1-4",
        "checkpoint_dir": "/content/GLAD/model/pill_4000step_bs32_eps_anomaly2_0/checkpoint-1500",
        "checkpoint_step": 1500,
        "output_name": "pill_4000step_bs32_eps_anomaly2_0",
        "denoise_step": 750,
        "input_threshold": 0.35,
        "v": 0,
        "min_step": 350,
        "resolution": 512,
        "test_batch_size": 2,
        "dino_resolution": 512,
        "inference_step": 25,
        "save_visuals": True,
    },
    "grid": {
        "enabled": True,
        "dataset_path": f"{DATA_ROOT}/grid",
        "pretrained_sd_path": f"{PRETRAIN_ROOT}/stable-diffusion-v1-4",
        "checkpoint_dir": "/content/GLAD/model/grid_4000step_bs32_eps_anomaly2_0/checkpoint-3000",
        "checkpoint_step": 3000,
        "output_name": "grid_4000step_bs32_eps_anomaly2_0",
        "denoise_step": 750,
        "input_threshold": 0.47,
        "v": 0,
        "min_step": 350,
        "resolution": 512,
        "test_batch_size": 2,
        "dino_resolution": 512,
        "inference_step": 25,
        "save_visuals": True,
    },
}

for c in CATEGORIES:
    print(c, GLAD_CONFIG[c]["enabled"], GLAD_CONFIG[c]["dataset_path"])


bottle True /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/bottle
cable True /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/cable
capsule True /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/capsule
pill True /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/pill
grid True /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/grid


In [6]:
# 3. GLAD clone（未 clone のときだけ）
if not os.path.exists(GLAD_REPO_DIR):
    !git clone https://github.com/hyao1/GLAD.git /content/GLAD
else:
    print("GLAD repo already exists:", GLAD_REPO_DIR)


GLAD repo already exists: /content/GLAD


In [7]:
# 4. 依存関係のインストール
# GLAD を Colab 上で動かすための固定版
!pip uninstall -y peft gradio gradio-client datasets sentence-transformers transformers diffusers accelerate huggingface-hub tokenizers || true
!pip install -q numpy==1.26.4 scipy==1.13.1 scikit-image==0.22.0 scikit-learn==1.4.2 imgaug==0.4.0
!pip install -q accelerate==0.27.2 diffusers==0.29.2 transformers==4.41.2 huggingface-hub==0.23.5 safetensors==0.4.3 einops==0.7.0 torchmetrics==1.4.0 kornia bitsandbytes
print("Install done. 必要なら Runtime restart 後に次セルから再開してください。")


Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: diffusers 0.29.2
Uninstalling diffusers-0.29.2:
  Successfully uninstalled diffusers-0.29.2
Found existing installation: accelerate 0.27.2
Uninstalling accelerate-0.27.2:
  Successfully uninstalled accelerate-0.27.2
Found existing installation: huggingface-hub 0.23.5
Uninstalling huggingface-hub-0.23.5:
  Successfully uninstalled huggingface-hub-0.23.5
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
Install done. 必要なら Runtime restart 後に次セルから再開してください。


In [8]:
# 5. import 確認
import numpy as np, scipy, skimage, sklearn, torch, diffusers, transformers, accelerate, safetensors, imgaug, kornia, bitsandbytes
print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

numpy: 1.26.4
torch: 2.10.0+cu128
cuda available: True
diffusers: 0.29.2
transformers: 4.41.2
accelerate: 0.27.2
gpu: Tesla T4


In [9]:
# 6. GLAD の randn_tensor import patch（Colab 用）
from pathlib import Path

targets = [
    Path("/content/GLAD/pipeline.py"),
    Path("/content/GLAD/ddim_scheduling.py"),
]

for path in targets:
    text = path.read_text()
    text = text.replace(
        "from diffusers.utils import randn_tensor",
        "from diffusers.utils.torch_utils import randn_tensor"
    )
    text = text.replace(
        "from diffusers.utils import (\n    deprecate,\n    is_accelerate_available,\n    is_accelerate_version,\n    logging,\n    randn_tensor,\n    replace_example_docstring,\n)",
        "from diffusers.utils import (\n    deprecate,\n    is_accelerate_available,\n    is_accelerate_version,\n    logging,\n    replace_example_docstring,\n)\nfrom diffusers.utils.torch_utils import randn_tensor"
    )
    text = text.replace(
        "from diffusers.utils import BaseOutput, randn_tensor",
        "from diffusers.utils import BaseOutput\nfrom diffusers.utils.torch_utils import randn_tensor"
    )
    path.write_text(text)
    print("patched:", path)


patched: /content/GLAD/pipeline.py
patched: /content/GLAD/ddim_scheduling.py


In [10]:
# 7. リポジトリ import 確認
import sys, importlib
sys.path.append("/content/GLAD")
modules_to_try = ["pipeline", "ddim_scheduling", "creat_model", "dataset.dataset", "dataset.transforms", "main"]
for m in modules_to_try:
    try:
        importlib.import_module(m)
        print("[OK]", m)
    except Exception as e:
        print("[NG]", m, type(e).__name__, e)


[OK] pipeline
[OK] ddim_scheduling
[OK] creat_model
[OK] dataset.dataset
[OK] dataset.transforms
[OK] main


In [11]:
# 8. Drive 上のファイル配置確認
import os, json

def check_glad_category_files(category, cfg):
    print("\n==", category, "==")
    checks = {
        "dataset_path": cfg["dataset_path"],
        "train_dir": os.path.join(cfg["dataset_path"], "train"),
        "test_dir": os.path.join(cfg["dataset_path"], "test"),
        "gt_dir": os.path.join(cfg["dataset_path"], "ground_truth"),
        "pretrained_sd_path": cfg["pretrained_sd_path"],
        "checkpoint_dir": cfg["checkpoint_dir"] if cfg["checkpoint_dir"] else "(unset)",
        "checkpoint_file": os.path.join(cfg["checkpoint_dir"], "pytorch_model.bin") if cfg["checkpoint_dir"] else "(unset)",
    }
    for k, v in checks.items():
        exists = os.path.exists(v) if isinstance(v, str) and not v.startswith("(") else False
        print(f"{k}: {v} -> {exists}")

for c in CATEGORIES:
    if GLAD_CONFIG[c]["enabled"]:
        check_glad_category_files(c, GLAD_CONFIG[c])



== bottle ==
dataset_path: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/bottle -> True
train_dir: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/bottle/train -> True
test_dir: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/bottle/test -> True
gt_dir: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/bottle/ground_truth -> True
pretrained_sd_path: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/stable-diffusion-v1-4 -> True
checkpoint_dir: /content/GLAD/model/bottle_4000step_bs32_eps_anomaly2_0/checkpoint-2500 -> False
checkpoint_file: /content/GLAD/model/bottle_4000step_bs32_eps_anomaly2_0/checkpoint-2500/pytorch_model.bin -> False

== cable ==
dataset_path: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/cable -> True
train_dir: /content/drive/MyDrive/industrial-anomaly-detection-app/data/MVTec-AD/cable/train -> True
test_dir: /content/drive/MyDrive/indu

In [12]:
# 9. main import と device 設定
import copy
import main
main.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("main.device =", main.device)


main.device = cuda


In [13]:
# 10. 共通 helper
import json
from torch.utils.data import DataLoader
from dataset.dataset import MVTecDataset

def build_glad_args(category, cfg):
    class Args:
        pass
    args = Args()
    args.train = False
    args.pretrained_model_name_or_path = cfg["pretrained_sd_path"]
    args.instance_data_dir = DATA_ROOT
    args.anomaly_data_dir = f"{BASE_DIR}/data/dtd"
    args.output_dir = os.path.join("/content/GLAD/model", cfg["output_name"]) if cfg["output_name"] else None
    args.checkpointing_steps = cfg["checkpoint_step"]
    args.instance_prompt = "a photo of sks"
    args.resolution = cfg["resolution"]
    args.test_batch_size = cfg["test_batch_size"]
    args.class_name = category
    args.pre_compute_text_embeddings = True
    args.seed = 0
    args.mixed_precision = "fp16"
    args.gradient_accumulation_steps = 1
    args.gradient_checkpointing = True
    args.use_8bit_adam = True
    args.learning_rate = 5e-6
    args.lr_scheduler = "constant"
    args.lr_warmup_steps = 0
    args.max_train_steps = cfg["checkpoint_step"] or 4000
    args.dataloader_num_workers = 0
    args.logging_dir = "logs"
    args.dino_resolution = cfg["dino_resolution"]
    args.inference_step = cfg["inference_step"]
    args.denoise_step = cfg["denoise_step"]
    args.input_threshold = cfg["input_threshold"]
    args.v = cfg["v"]
    args.min_step = cfg["min_step"]
    args.save_path = None
    args.output_name = cfg["output_name"]
    return args

def save_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    print("saved:", path)


In [14]:
# 11. 単カテゴリ評価関数
import numpy as np

def evaluate_glad_category(category, cfg):
    args = build_glad_args(category, cfg)
    dino_model, val_pipe = main.load_test_model(args, torch.float16)
    dino_frozen = copy.deepcopy(dino_model)

    val_dataset = MVTecDataset(
        instance_data_root=DATA_ROOT,
        instance_prompt=args.instance_prompt,
        class_name=category,
        tokenizer=None,
        resize=args.resolution,
        img_size=args.resolution,
        train=False
    )
    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.test_batch_size,
        num_workers=0,
        shuffle=False,
        pin_memory=True,
    )

    results = main.test(
        dino_model=dino_model,
        dino_frozen=dino_frozen,
        val_pipe=val_pipe,
        weight_dtype=torch.float16,
        val_dataloader=val_dataloader,
        args=args,
        device=main.device,
        class_name=category,
        checkpoint_step=cfg["checkpoint_step"],
    )

    metrics = {
        "Method": "GLAD",
        "Dataset": "MVTec-AD",
        "Category": category,
        "I_AUROC": float(results[0]),
        "I_AP": float(results[1]),
        "I_F1max": float(results[2]),
        "P_AUROC": float(results[3]),
        "P_AP": float(results[4]),
        "P_F1max": float(results[5]),
        "PRO": float(results[6]),
        "checkpoint_step": cfg["checkpoint_step"],
        "denoise_step": cfg["denoise_step"],
        "input_threshold": cfg["input_threshold"],
        "v": cfg["v"],
        "min_step": cfg["min_step"],
    }

    save_path = f"{COMPARISON_ROOT}/GLAD/{category}/metrics.json"
    save_json(save_path, metrics)
    return metrics


In [15]:
# 12. Drive上のファイルをColab側にコピー

import os
import shutil

base_drive = "/content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/GLAD"
base_local = "/content/GLAD/model"

for category, cfg in GLAD_CONFIG.items():
    output_name = cfg["output_name"]
    checkpoint_step = cfg["checkpoint_step"]

    if output_name is None or checkpoint_step is None:
        print(f"[SKIP] {category}: config incomplete")
        continue

    src_dir = os.path.join(base_drive, output_name, f"checkpoint-{checkpoint_step}")
    src_file = os.path.join(src_dir, "pytorch_model.bin")

    dst_dir = os.path.join(base_local, output_name, f"checkpoint-{checkpoint_step}")
    dst_file = os.path.join(dst_dir, "pytorch_model.bin")

    print(f"\n=== {category} ===")
    print("src:", src_file)
    print("src exists:", os.path.exists(src_file))

    if not os.path.exists(src_file):
        print(f"[SKIP] missing checkpoint on Drive for {category}")
        continue

    os.makedirs(dst_dir, exist_ok=True)
    shutil.copy2(src_file, dst_file)

    print("copied to:", dst_file)
    print("dst exists:", os.path.exists(dst_file))
    print("size(bytes):", os.path.getsize(dst_file))


=== bottle ===
src: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/GLAD/bottle_4000step_bs32_eps_anomaly2_0/checkpoint-2500/pytorch_model.bin
src exists: True
copied to: /content/GLAD/model/bottle_4000step_bs32_eps_anomaly2_0/checkpoint-2500/pytorch_model.bin
dst exists: True
size(bytes): 3438369029

=== cable ===
src: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/GLAD/cable_4000step_bs32_eps_anomaly2_0/checkpoint-1500/pytorch_model.bin
src exists: True
copied to: /content/GLAD/model/cable_4000step_bs32_eps_anomaly2_0/checkpoint-1500/pytorch_model.bin
dst exists: True
size(bytes): 3438369029

=== capsule ===
src: /content/drive/MyDrive/industrial-anomaly-detection-app/pretrained/GLAD/capsule_4000step_bs32_eps_anomaly2_0/checkpoint-2000/pytorch_model.bin
src exists: True
copied to: /content/GLAD/model/capsule_4000step_bs32_eps_anomaly2_0/checkpoint-2000/pytorch_model.bin
dst exists: True
size(bytes): 3438369029

=== pill ===
src: /content/drive/

In [16]:
# 13. 評価実行（enabled=True のカテゴリだけ）
glad_results = []
for category in CATEGORIES:
    cfg = GLAD_CONFIG[category]

    required = [cfg["checkpoint_dir"], cfg["output_name"], cfg["checkpoint_step"], cfg["denoise_step"], cfg["input_threshold"], cfg["v"], cfg["min_step"]]
    if any(v is None for v in required):
        print("skip (config incomplete):", category)
        continue
    metrics = evaluate_glad_category(category, cfg)
    glad_results.append(metrics)

glad_results


Downloading: "https://dl.fbaipublicfiles.com/dino/dino_vitbase8_pretrain/dino_vitbase8_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dino_vitbase8_pretrain.pth


100%|██████████| 327M/327M [00:06<00:00, 54.0MB/s]
The config attributes {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

test:bottle, checkpoint_step:2500
test-------- I-AUROC/I-AP/I-F1-max/P-AUROC/P-AP/P-F1-max/PRO:1.0/1.0/1.0/0.9871/0.855/0.7868/0.9615-----
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/bottle/metrics.json


The config attributes {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


...

test:pill, checkpoint_step:1500
test-------- I-AUROC/I-AP/I-F1-max/P-AUROC/P-AP/P-F1-max/PRO:0.9869/0.9976/0.9722/0.979/0.7599/0.708/0.976-----
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/pill/metrics.json


The config attributes {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

test:grid, checkpoint_step:3000
test-------- I-AUROC/I-AP/I-F1-max/P-AUROC/P-AP/P-F1-max/PRO:1.0/1.0/1.0/0.9957/0.5467/0.5576/0.9833-----
saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/grid/metrics.json


[{'Method': 'GLAD',
  'Dataset': 'MVTec-AD',
  'Category': 'bottle',
  'I_AUROC': 100.0,
  'I_AP': 100.0,
  'I_F1max': 100.0,
  'P_AUROC': 98.71,
  'P_AP': 85.5,
  'P_F1max': 78.68,
  'PRO': 96.15,
  'checkpoint_step': 2500,
  'denoise_step': 750,
  'input_threshold': 0.32,
  'v': 0,
  'min_step': 350},
 {'Method': 'GLAD',
  'Dataset': 'MVTec-AD',
  'Category': 'cable',
  'I_AUROC': 99.19,
  'I_AP': 99.56,
  'I_F1max': 97.21,
  'P_AUROC': 98.11999999999999,
  'P_AP': 68.61,
  'P_F1max': 66.05,
  'PRO': 94.01,
  'checkpoint_step': 1500,
  'denoise_step': 750,
  'input_threshold': 0.4,
  'v': 0,
  'min_step': 350},
 {'Method': 'GLAD',
  'Dataset': 'MVTec-AD',
  'Category': 'capsule',
  'I_AUROC': 98.48,
  'I_AP': 99.66000000000001,
  'I_F1max': 97.76,
  'P_AUROC': 98.54,
  'P_AP': 53.669999999999995,
  'P_F1max': 54.279999999999994,
  'PRO': 94.24,
  'checkpoint_step': 2000,
  'denoise_step': 600,
  'input_threshold': 0.4,
  'v': 0,
  'min_step': 350},
 {'Method': 'GLAD',
  'Dataset': 'M

In [17]:
# 14. 集計 CSV 作成
import pandas as pd
if glad_results:
    glad_df = pd.DataFrame(glad_results)
    out_csv = f"{COMPARISON_ROOT}/tables/glad_multi_metrics.csv"
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    glad_df.to_csv(out_csv, index=False)
    print("saved:", out_csv)
    display(glad_df)
else:
    print("No GLAD results yet.")


saved: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/tables/glad_multi_metrics.csv


,Method,Dataset,Category,I_AUROC,I_AP,I_F1max,P_AUROC,P_AP,P_F1max,PRO,checkpoint_step,denoise_step,input_threshold,v,min_step
0,GLAD,MVTec-AD,bottle,100.00,100.00,100.00,98.71,85.50,78.68,96.15,2500,750,0.32,0,350
1,GLAD,MVTec-AD,cable,99.19,99.56,97.21,98.12,68.61,66.05,94.01,1500,750,0.40,0,350
2,GLAD,MVTec-AD,capsule,98.48,99.66,97.76,98.54,53.67,54.28,94.24,2000,600,0.40,0,350
3,GLAD,MVTec-AD,pill,98.69,99.76,97.22,97.90,75.99,70.80,97.60,1500,750,0.35,0,350
4,GLAD,MVTec-AD,grid,100.00,100.00,100.00,99.57,54.67,55.76,98.33,3000,750,0.47,0,350


In [18]:
from pathlib import Path

path = Path("/content/GLAD/main.py")
text = path.read_text()

text = text.replace(
    "masks = np.array(masks, dteype=np.int_)",
    "masks = np.array(masks, dtype=np.int_)",
)

path.write_text(text)
print("patched:", path)

patched: /content/GLAD/main.py


In [19]:
from pathlib import Path

path = Path("/content/GLAD/main.py")
text = path.read_text()

text = text.replace(
    "heat1 = utilize.apply_ad_scoremap(sources[i] * 255, heat[i])",
    "heat1 = apply_ad_scoremap(sources[i] * 255, heat[i])"
)

path.write_text(text)
print("patched:", path)

patched: /content/GLAD/main.py


In [20]:
import importlib
import main

importlib.reload(main)
print("reloaded main")

reloaded main


In [21]:
# GLAD 可視化画像保存セル
import os
import shutil
import torch
from torch.utils.data import DataLoader

SAVE_GLAD_SAMPLES = True
NUM_GLAD_SAMPLES_PER_CATEGORY = 2

for category in CATEGORIES:
    cfg = GLAD_CONFIG[category]
    args = build_glad_args(category, cfg)
    dino_model, val_pipe = main.load_test_model(args, torch.float16)
    dino_frozen = copy.deepcopy(dino_model)

    if not cfg.get("enabled", True):
        print("skip:", category)
        continue

    required = [
        cfg.get("checkpoint_dir"),
        cfg.get("output_name"),
        cfg.get("checkpoint_step"),
        cfg.get("denoise_step"),
        cfg.get("input_threshold"),
        cfg.get("v"),
        cfg.get("min_step"),
    ]
    if any(v is None for v in required):
        print("skip (config incomplete):", category)
        continue

    print(f"\n===== GLAD visualization: {category} =====")

    # 1) 評価時と同じ設定を入れる
    args.class_name = category
    args.output_dir = os.path.join("/content/GLAD/model", cfg["output_name"])
    args.denoise_step = cfg["denoise_step"]
    args.input_threshold = cfg["input_threshold"]
    args.v = cfg["v"]
    args.min_step = cfg["min_step"]

    # 2) dataset / dataloader
    val_dataset = MVTecDataset(
        instance_data_root=args.instance_data_dir,
        instance_prompt=args.instance_prompt,
        class_name=category,
        tokenizer=None,
        resize=args.resolution,
        img_size=args.resolution,
        train=False
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.test_batch_size,
        num_workers=0,
        shuffle=False,
        pin_memory=True,
    )

    # 3) visual 実行
    _ = main.visual(
        dino_model=dino_model,
        dino_frozen=dino_frozen,
        val_pipe=val_pipe,
        weight_dtype=torch.float16,
        val_dataloader=val_dataloader,
        args=args,
        device=main.device,
        class_name=category,
        checkpoint_step=cfg["checkpoint_step"],
    )

    # 4) GLAD が保存したディレクトリを特定
    src_dir = f"{args.inference_step}NFE_{args.denoise_step}step_lcm/{args.output_dir.split('/')[-1]}_{args.input_threshold}"
    print("src_dir:", src_dir, "exists:", os.path.exists(src_dir))

    if not os.path.exists(src_dir):
        print(f"[WARN] visualization dir not found for {category}")
        continue

    # 5) 保存先
    dst_dir = f"{COMPARISON_ROOT}/GLAD/{category}/samples"
    os.makedirs(dst_dir, exist_ok=True)

    # 6) スコア順で並んでいる想定なので先頭から数枚コピー
    files = sorted([
        f for f in os.listdir(src_dir)
        if os.path.isfile(os.path.join(src_dir, f))
    ])[:NUM_GLAD_SAMPLES_PER_CATEGORY]

    print("copy files:", files)

    for f in files:
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))

    print("saved to:", dst_dir)

The config attributes {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]


===== GLAD visualization: bottle =====
test:bottle, checkpoint_step:2500
test-------- I-AUROC/I-AP/I-F1-max/P-AUROC/P-AP/P-F1-max/PRO:0.9976/0.9993/0.992/0.9848/0.8484/0.7859/0.9593-----
src_dir: 25NFE_750step_lcm/bottle_4000step_bs32_eps_anomaly2_0_0.32 exists: True
copy files: ['0.0000_331_good_004.png', '0.0100_331_good_013.png']
saved to: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/bottle/samples


The config attributes {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

...

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]


===== GLAD visualization: pill =====
test:pill, checkpoint_step:1500
test-------- I-AUROC/I-AP/I-F1-max/P-AUROC/P-AP/P-F1-max/PRO:0.9744/0.9953/0.9675/0.9791/0.7588/0.7046/0.9761-----
src_dir: 25NFE_750step_lcm/pill_4000step_bs32_eps_anomaly2_0_0.35 exists: True
copy files: ['0.0000_601_good_004.png', '0.0151_601_good_001.png']
saved to: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/pill/samples


The config attributes {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} were passed to StableDiffusionPipeline, but are not expected and will be ignored. Please verify your model_index.json configuration file.
Keyword arguments {'safety_checker': ['stable_diffusion', 'StableDiffusionSafetyChecker']} are not expected by StableDiffusionPipeline and will be ignored.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]


===== GLAD visualization: grid =====
test:grid, checkpoint_step:3000
test-------- I-AUROC/I-AP/I-F1-max/P-AUROC/P-AP/P-F1-max/PRO:1.0/1.0/1.0/0.9956/0.5434/0.5564/0.9832-----
src_dir: 25NFE_750step_lcm/grid_4000step_bs32_eps_anomaly2_0_0.47 exists: True
copy files: ['0.0000_331_good_011.png', '0.0175_331_good_018.png']
saved to: /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/grid/samples


In [22]:
# GLAD 保存確認
for category in CATEGORIES:
    p = f"{COMPARISON_ROOT}/GLAD/{category}/samples"
    print("\n", category, "->", p, os.path.exists(p))
    if os.path.exists(p):
        print(os.listdir(p)[:5])


 bottle -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/bottle/samples True
['0.0000_331_good_004.png', '0.0100_331_good_013.png']

 cable -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/cable/samples True
['0.0000_331_good_023.png', '0.0357_541_good_042.png']

 capsule -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/capsule/samples True
['0.0000_337_good_005.png', '0.0034_337_good_001.png']

 pill -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/pill/samples True
['0.0000_601_good_004.png', '0.0151_601_good_001.png']

 grid -> /content/drive/MyDrive/industrial-anomaly-detection-app/comparison/GLAD/grid/samples True
['0.0000_331_good_011.png', '0.0175_331_good_018.png']
